# 卵巢癌 Xenium + Flex scRNA-seq 数据预处理

**数据集**
- scRNA-seq：Xenium Flex 平台，`17k_Ovarian_Cancer_scFFPE_count_filtered_feature_bc_matrix.h5`
- 预标注：`FLEX_Ovarian_Barcode_Cluster_Annotation.csv`
- Xenium：`Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs/`

**与肾脏流水线的核心区别**
| 步骤 | 肾脏 | 卵巢 |
|---|---|---|
| scRNA 聚类 | 需要（Harmony + FindClusters） | **跳过**（Flex 已提供 CSV 注释） |
| 手动注释 | 需要（看 DotPlot） | **跳过** |
| Seurat 标签转移 | 需要（k.weight 兜底）| **跳过** |
| R notebook 总耗时 | ~2 小时 | **~10 分钟** |

⚠️ **此 notebook 仅用于生成 `.rds` 缓存供可视化。**  
Python 训练 notebook 的 Cell 2 已改为直接用 `scanpy` 加载，**不依赖此 R notebook 的输出**。

In [ ]:
# ============================================================
# Cell 1: 初始化
# ============================================================
setwd("/home/ailab/caohao/AdaDiss/")

suppressPackageStartupMessages({
    library(Seurat)
    library(dplyr)
    library(ggplot2)
    library(Matrix)
})

PATHS <- list(
    raw = list(
        flex_h5     = "data/raw/flex/17k_Ovarian_Cancer_scFFPE_count_filtered_feature_bc_matrix.h5",
        flex_annot  = "data/raw/flex/FLEX_Ovarian_Barcode_Cluster_Annotation.csv",
        xenium_dir  = "data/raw/xenium/Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs/"
    ),
    cache = list(
        scrna_annotated = "data/cache/scrna/scrna_annotated_ovarian.rds",
        xenium_base     = "data/cache/xenium/xenium_base_ovarian.rds"
    )
)

for (d in c("data/cache/scrna", "data/cache/xenium")) {
    dir.create(d, recursive = TRUE, showWarnings = FALSE)
}

cat("=== 路径检查 ===\n")
cat("Flex H5   :", file.exists(PATHS$raw$flex_h5),    "\n")
cat("Flex CSV  :", file.exists(PATHS$raw$flex_annot), "\n")
cat("Xenium dir:", dir.exists(PATHS$raw$xenium_dir),  "\n")

In [ ]:
# ============================================================
# Cell 2: 加载 Flex scRNA-seq + 附加预标注
# Flex 平台已提供 cluster annotation CSV，跳过聚类步骤
# ============================================================

if (file.exists(PATHS$cache$scrna_annotated)) {
    cat("[Cache HIT] 加载已注释 scRNA 对象...\n")
    scrna_obj <- readRDS(PATHS$cache$scrna_annotated)
} else {
    cat("[Cache MISS] 加载 Flex H5 文件...\n")

    # ── 加载计数矩阵 ───────────────────────────────────────
    counts <- Read10X_h5(PATHS$raw$flex_h5)
    scrna_obj <- CreateSeuratObject(
        counts   = counts,
        project  = "Ovarian_Flex",
        min.cells = 3, min.features = 200
    )
    cat("  原始细胞数:", ncol(scrna_obj), "\n")
    cat("  基因数    :", nrow(scrna_obj), "\n")

    # ── 附加预标注（直接从 Flex Barcode CSV）─────────────────
    annot <- read.csv(PATHS$raw$flex_annot, stringsAsFactors = FALSE)
    cat("  CSV 列名  :", paste(colnames(annot), collapse = ", "), "\n")
    cat("  CSV 前3行 :\n")
    print(head(annot, 3))

    # 自动识别 barcode 列 和 cell_type 列
    bc_col  <- grep("barcode|cell_id|barcodes", colnames(annot), 
                    ignore.case = TRUE, value = TRUE)[1]
    ct_col  <- grep("cell_type|celltype|annotation|cluster_name|label",
                    colnames(annot), ignore.case = TRUE, value = TRUE)[1]
    cat("  barcode 列:", bc_col, "\n")
    cat("  类型列    :", ct_col, "\n")

    rownames(annot) <- annot[[bc_col]]
    common_bc <- intersect(colnames(scrna_obj), annot[[bc_col]])
    cat("  有标注细胞:", length(common_bc), "/", ncol(scrna_obj), "\n")

    scrna_obj <- scrna_obj[, common_bc]
    scrna_obj$cell_type <- annot[common_bc, ct_col]

    # ── 基础 QC（线粒体过滤）─────────────────────────────────
    scrna_obj[["percent.mt"]] <- PercentageFeatureSet(scrna_obj, pattern = "^MT-")
    scrna_obj <- subset(scrna_obj,
        subset = nFeature_RNA > 200 & nFeature_RNA < 6000 & percent.mt < 25
    )
    cat("  QC 后细胞数:", ncol(scrna_obj), "\n")

    # ── 标准化（供 UMAP 可视化，不影响 GNN 输入）────────────
    scrna_obj <- NormalizeData(scrna_obj, verbose = FALSE)
    scrna_obj <- FindVariableFeatures(scrna_obj, nfeatures = 3000, verbose = FALSE)
    scrna_obj <- ScaleData(scrna_obj, verbose = FALSE)
    scrna_obj <- RunPCA(scrna_obj, npcs = 30, verbose = FALSE)
    scrna_obj <- RunUMAP(scrna_obj, dims = 1:20, verbose = FALSE)

    saveRDS(scrna_obj, PATHS$cache$scrna_annotated)
    cat("[Saved] scRNA 对象 →", PATHS$cache$scrna_annotated, "\n")
}

cat("\n=== 细胞类型分布 ===\n")
print(sort(table(scrna_obj$cell_type), decreasing = TRUE))

In [ ]:
# ============================================================
# Cell 3: UMAP 可视化（检查注释质量）
# ============================================================

p <- DimPlot(scrna_obj, group.by = "cell_type", label = TRUE, repel = TRUE,
             pt.size = 0.3) +
     ggtitle("Flex scRNA-seq — Ovarian Cancer Cell Types") +
     theme(plot.title = element_text(hjust = 0.5))

print(p)
ggsave("figures/scrna_umap_ovarian.pdf", p, width = 10, height = 8)

In [ ]:
# ============================================================
# Cell 4: 加载 Xenium 数据（提取 gene panel 供 Python 使用）
# ============================================================

if (file.exists(PATHS$cache$xenium_base)) {
    cat("[Cache HIT] 加载 Xenium 基础缓存...\n")
    xenium_obj <- readRDS(PATHS$cache$xenium_base)
} else {
    cat("[Cache MISS] 加载 Xenium 数据...\n")
    xenium_obj <- LoadXenium(
        data.dir = PATHS$raw$xenium_dir,
        fov       = "fov"
    )
    xenium_obj <- subset(xenium_obj, nCount_Xenium > 0)
    saveRDS(xenium_obj, PATHS$cache$xenium_base)
    cat("[Saved] Xenium 基础对象 →", PATHS$cache$xenium_base, "\n")
}

cat("  Xenium 细胞数:", ncol(xenium_obj), "\n")
cat("  Xenium 基因数:", nrow(xenium_obj), "\n")

# 共同基因（Python 流水线需要）
common_genes <- intersect(rownames(scrna_obj), rownames(xenium_obj))
cat("  共同基因数  :", length(common_genes), "\n")

In [ ]:
# ============================================================
# Cell 5: 完成确认
# ============================================================

cat("=== 卵巢癌 R 预处理完成 ===\n\n")
cat("缓存文件状态：\n")
for (nm in names(PATHS$cache)) {
    f    <- PATHS$cache[[nm]]
    ex   <- file.exists(f)
    size <- if (ex) paste0(round(file.size(f)/1e6, 1), " MB") else "不存在"
    cat(sprintf("  %-28s %s  (%s)\n", nm, if (ex) "✅" else "❌", size))
}

cat("\n⚠️  注意：Python 训练 notebook 的 Cell 2 已改为直接用 scanpy 加载，")
cat("\n    不依赖本 notebook 的 .rds 输出，可直接运行 Python notebook。\n")